# Week 3: Exercise 7 - Tool Registry

**Goal:** Build a registry that manages all tools in one place.

No API key needed for this exercise.


## Step 1: The Tool Dataclass


In [1]:
from typing import Dict, List, Callable
from dataclasses import dataclass

@dataclass
class Tool:
    name: str
    description: str
    parameters: dict
    handler: Callable
    source: str = "builtin"
    category: str = "general"


## Step 2: Implement the ToolRegistry

The registry maps tool names to `Tool` objects. Implement:

- `register(...)` — create a `Tool`, store it, track its category
- `get_schema(name)` — return the OpenAI tool-schema dict for one tool
- `get_all_schemas()` — schemas for every registered tool
- `call(name, args)` — execute a tool's handler with `**args`


In [2]:
class ToolRegistry:
    """TODO: Implement the registry."""

    def __init__(self):
        self._tools: Dict[str, Tool] = {}
        self._categories: Dict[str, List[str]] = {}

    def register(self, name, description, parameters, handler, category="general"):
        """TODO: Register a tool."""
        # Hint: 1. Create a Tool object using the arguments
        #       2. Store it: self._tools[name] = tool
        #       3. Track category: if category not in self._categories, create empty list,
        #          then append the name (OUTSIDE the if!)
        tool = Tool(name, description, parameters, handler, category)
        self._tools[name] = tool
        if category not in self._categories:
            self._categories[category] = []

        self._categories[category].append(name)

    def get_schema(self, name):
        """TODO: Get OpenAI-format schema for one tool."""
        # Hint: look up tool = self._tools.get(name)
        #       return {"type": "function", "function": {name, description, parameters}}
        tool = self._tools[name]
        return {
            "type": "function",
            "function": {
                "name": name,
                "description": tool.description,
                "parameters": tool.parameters
            }
        }

    def get_all_schemas(self):
        """TODO: Get schemas for all tools."""
        # Hint: [self.get_schema(name) for name in self._tools]
        return [self.get_schema(name) for name in self._tools.key()]

    def call(self, name, args):
        """TODO: Execute a tool by name."""
        # Hint: look up tool, raise ValueError if missing,
        #       return tool.handler(**args)
        try:
            if name in self._tools.key():
                return self._tools[name].handler(**args)
            else: raise ValueError(f"Function {name} not found!")
        except TypeError as TE:
            return(f"Errror: {TE}")

    def list_tools(self):
        return list(self._tools.keys())

    def list_by_category(self):
        return dict(self._categories)


## Step 3: Simulated MCP Server


In [3]:
class MCPServer:
    """Simulated MCP server connection."""
    def __init__(self, name, command):
        self.name = name
        self.command = command
        self.connected = False

    async def connect(self):
        print(f"Connecting to MCP server: {self.name}...")
        self.connected = True

    def disconnect(self):
        print(f"Disconnected from {self.name}")
        self.connected = False


In [4]:
# Test the MCP server
# NOTE: connect() is async. In Jupyter we can use await directly,
# or asyncio.run() in a plain script.

server = MCPServer("weather-tools", "python weather_server.py")

# Test 1: starts disconnected
assert server.connected == False, "Should start disconnected"
print(f"  Created: {server.name}")
print(f"  Connected before: {server.connected}")

# Test 2: connect() sets connected=True
# await works directly in Jupyter cells:
await server.connect()
assert server.connected == True, "Should be connected after connect()"
print(f"  Connected after: {server.connected}")

# Test 3: disconnect() sets connected=False
server.disconnect()
assert server.connected == False, "Should be disconnected"
print(f"  Connected after disconnect: {server.connected}")
print("MCP test passed!")


  Created: weather-tools
  Connected before: False
Connecting to MCP server: weather-tools...
  Connected after: True
Disconnected from weather-tools
  Connected after disconnect: False
MCP test passed!


## Step 4: Register Tools and Test
**TODO:** Register two tools and verify your registry works.


In [5]:
registry = ToolRegistry()

def get_time():
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

def add_numbers(a: int, b: int):
    return a + b

# TODO: register get_time and add_numbers with proper JSON schema parameters
registry.register("get_time", "Get the current time",
    {"type": "object", "properties": {}}, get_time, category="utility")
registry.register("add_numbers", "Add two numbers",
    {"type": "object", "properties": {
        "a": {"type": "integer"}, "b": {"type": "integer"}
    }, "required": ["a", "b"]}, add_numbers, category="math")

print("All tools:", registry.list_tools())
print("Categories:", registry.list_by_category())


All tools: ['get_time', 'add_numbers']
Categories: {'utility': ['get_time'], 'math': ['add_numbers']}


## Key Takeaways
- The registry decouples tool definition from tool usage
- `get_all_schemas()` is what you pass to `tools=` in the LLM API call
